# Check the CSV dataset before training:
-> check the row, col, dtypes, etc.

In [1]:
import pandas as pd
import chess
import torch
from torch.utils.data import Dataset

df = pd.read_csv("Lichess_clean_2016_80k_games.csv")
print("Rows:", len(df))
df.head()


Rows: 79788


,White,Black,Result,ECO,Opening,Moves,MoveCount,WhiteWin,BlackWin,Draw
0,Mescalero25,fialho,1-0,C00,French Defense: King's Indian Attack,e4 e6 d3 d5 Nd2 dxe4 Nxe4 b6 g3 Bb7 Bg2 Be7 Nf...,77,1,0,0
1,theosis101,sebafreire,1-0,A43,Old Benoni Defense,d4 c5 c4 cxd4 Nf3 Nc6 e3 e6 exd4 Nf6 Nc3 d5 cx...,55,1,0,0
2,BrettDale,Viriskensoshir,0-1,B72,"Sicilian Defense: Dragon, Classical Attack",e4 c5 Nf3 d6 d4 cxd4 Nxd4 Nf6 Nc3 g6 Be2 Bg7 B...,54,0,1,0
3,borgs,IntiTupac,0-1,A13,English Opening: Agincourt Defense #3,c4 e6 Nf3 a6 g3 g6 Bg2 Bg7 O-O Ne7 d4 d5 cxd5 ...,92,0,1,0
4,Aquariano,Moszkowski,1-0,A40,Englund Gambit Complex: Soller Gambit,d4 e5 dxe5 f6 exf6 Nxf6 a3 Bc5 e3 O-O Nc3 a6 B...,119,1,0,0


# Dataset → (board → next_move)

In [2]:
class ChessDataset(Dataset):
    def __init__(self, df, max_games=5000):
        self.samples = []
        for i, row in enumerate(df.itertuples()):
            board = chess.Board()
            moves = str(row.Moves).split()
            for j in range(len(moves)-1):
                try:
                    move = board.parse_san(moves[j])
                    board.push(move)
                    self.samples.append((self.board_to_tensor(board), moves[j+1]))
                except Exception:
                    break
            if i % 200 == 0:
                print(f"{i}/{max_games} games processed...")
            if i >= max_games:
                break

    def board_to_tensor(self, board):
        tensor = torch.zeros((12, 8, 8), dtype=torch.float32)
        piece_map = {'P':0,'N':1,'B':2,'R':3,'Q':4,'K':5,
                     'p':6,'n':7,'b':8,'r':9,'q':10,'k':11}
        for sq, pc in board.piece_map().items():
            x, y = divmod(sq, 8)
            tensor[piece_map[pc.symbol()], x, y] = 1
            return tensor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]
